# Demoparser exploration

This notebook serves as an exploration stage in demo data parsing (.dem).

In [ ]:
from demoparser2 import DemoParser

sync_fields = ["round_start_time",
               "is_freeze_period",
               "round_in_progress",
               "total_rounds_played",
               "game_time",]

boundary_fields = ["is_warmup_period",
                   "round_win_status",
                   "num_player_alive_ct",
                   "num_player_alive_t",]

position_fields = ["X", "Y", "Z",
                   "player_steamid",
                   "player_name",
                   "is_alive",
                   "team_name"]

fields_to_parse = position_fields + sync_fields

parser = DemoParser("../data/raw/demos/match_1.dem")
df = parser.parse_ticks(fields_to_parse)
print(df[[
    "tick",
    "total_rounds_played",
    "is_freeze_period",
    "game_time"
]].head(50))

         tick  total_rounds_played  is_freeze_period    game_time
108170  10819                    2             False  2118.390625
108171  10819                    2             False  2118.390625
108172  10819                    2             False  2118.390625
108173  10819                    2             False  2118.390625
108174  10819                    2             False  2118.390625
108175  10819                    2             False  2118.390625
108176  10819                    2             False  2118.390625
108177  10819                    2             False  2118.390625
108178  10819                    2             False  2118.390625
108179  10819                    2             False  2118.390625
108180  10820                    2             False  2118.406250
108181  10820                    2             False  2118.406250
108182  10820                    2             False  2118.406250
108183  10820                    2             False  2118.406250
108184  10

In [ ]:
round_number = 1 # set to loop in actual script

round_df = df[df["total_rounds_played"] == round_number]

round_start_tick = round_df[round_df["is_freeze_period"] == False].iloc[0]['tick']

round_end_tick = round_df["tick"].max()

print(round_start_tick)
print(round_end_tick)

round_2_data = {
  'start_tick': round_start_tick,
  'end_tick': round_end_tick,
  'tick_rate': 64,
  'duration_ticks': round_end_tick - round_start_tick,
  'duration_seconds': (round_end_tick - round_start_tick) / 64,
}

print(round_2_data)

11280
22326
{'start_tick': np.int32(11280), 'end_tick': np.int32(22326), 'tick_rate': 64, 'duration_ticks': np.int32(11046), 'duration_seconds': np.float64(172.59375)}


In [39]:
video_round_start = 0.0  # seconds
video_round_end = 172.5   # seconds

def video_time_to_tick(video_time):
  tick = round_2_data['start_tick'] + (video_time - video_round_start) / (video_round_end - video_round_start) * (round_2_data['end_tick'] - round_2_data['start_tick'])
  return int(tick)

In [41]:
round_positions = df[
  (df['total_rounds_played'] == 1) &
  (df['is_alive'] == True)
][['tick', 'X', 'Y', 'Z', 'player_steamid', 'player_name']]

round_positions.to_parquet('../data/sample/match1_round2_positions.parquet')

metadata = {
  'round_number': round_number,
  'start_tick': round_start_tick,
  'end_tick': round_end_tick,
  'tick_rate': 64,
  'video_file': 'round-2.mp4',
  'video_round_start': video_round_start,
  'video_round_end': video_round_end
}